# `cvit_cub.py` as a notebook

This notebook follows the repo's training script, `cvit_cub.py`, one piece at a time. The script is short: it defines the command-line
options, builds the model and data, trains, and tests. Here each piece is its own cell, with a note on what it does, so you can
see and change every step instead of running one command.

The first part (setup and data) is the same preparation as in my main notebook. The second part is the script itself.
Set the runtime to a GPU (`Runtime → Change runtime type`), then `Runtime → Run all`.

## Part 1: Setup and data (same as the main notebook)

Settings first. This is the only cell to edit. These are the repo's default training settings, except batch size 64.

In [ ]:
# ------------------------------------------------------------------
# Settings. This is the only cell you should need to edit.
# ------------------------------------------------------------------
import os, sys

REPO_URL = "https://github.com/IBM/concept_transformer"

IN_COLAB = "google.colab" in sys.modules
WORK_DIR = "/content" if IN_COLAB else os.path.expanduser("~/ctc_work")
os.makedirs(WORK_DIR, exist_ok=True)

REPO_DIR = f"{WORK_DIR}/concept_transformer"
DATA_DIR = f"{WORK_DIR}/data/cub2011"      # goes to --data_dir; the images land in DATA_DIR/CUB_200_2011/images

# How long / how hard to train. The repo's own defaults are 50 epochs, batch 32, warmup 10, lr 5e-5.
MAX_EPOCHS    = 50      # the repo's default, and what the paper trains for
BATCH_SIZE    = 64      # the repo default is 32; the A100 has plenty of memory for double that
WARMUP        = 10      # repo default (a fifth of a 50-epoch run)
LEARNING_RATE = 5e-5    # repo default
NUM_WORKERS   = 8 if IN_COLAB else 0

DEBUG = False           # True runs a single batch just to check the plumbing. Handy before a long run.

# Colab wipes everything when the runtime ends, so I copy the results to Google Drive at the end.
SAVE_TO_DRIVE   = True
DRIVE_OUT_DIR   = "/content/drive/MyDrive/concept_transformer_run"
AUTO_DISCONNECT = False  # flip to True to release the A100 automatically once the results are safely on Drive

### Clone the repo, install packages, patch it for modern PyTorch

The repo is from 2021, so it needs a few compatibility patches to run on today's Colab (the table in my main notebook lists each one).

In [ ]:
import os

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log -1 --oneline

In [ ]:
!pip -q install pytorch-lightning==1.9.5 torchmetrics==0.11.4 timm==0.4.12 albumentations==1.4.24 tensorboard

In [ ]:
import re, pathlib

def patch(path, pairs, regex=False):
    p = pathlib.Path(path); s = p.read_text(); orig = s
    for a, b in pairs:
        s = re.sub(a, b, s) if regex else s.replace(a, b)
    if s != orig:
        p.write_text(s); print("patched", path)
    else:
        print("already patched / no change", path)

# ctc_model.py: no pl_bolts, no apex, and drop the Trainer arguments Lightning 1.7 removed
patch("ctc/ctc_model.py", [
    ("from pl_bolts.optimizers.lr_scheduler import LinearWarmupCosineAnnealingLR\n",
     """import math
from torch.optim.lr_scheduler import LambdaLR


def LinearWarmupCosineAnnealingLR(optimizer, warmup_epochs, max_epochs, warmup_start_lr=0.0, eta_min=0.0):
    # Same schedule pl_bolts uses: linear warm-up, then cosine decay, stepped once per epoch.
    base = optimizer.param_groups[0]["lr"]
    def f(epoch):
        if warmup_epochs > 0 and epoch < warmup_epochs:
            return (warmup_start_lr + (base - warmup_start_lr) * epoch / max(1, warmup_epochs - 1)) / base
        t = (epoch - warmup_epochs) / max(1, max_epochs - warmup_epochs)
        return (eta_min + 0.5 * (base - eta_min) * (1 + math.cos(math.pi * t))) / base
    return LambdaLR(optimizer, f)
"""),
    ('kwargs = {"amp_backend": "apex", "amp_level": "O2", "precision": 16}', 'kwargs = {"precision": 16}'),
    ("            weights_summary=\"full\",\n", ""),
    ("                progress_bar_refresh_rate=10,\n", ""),
    ("            progress_bar_refresh_rate=10,\n", ""),
])

# Newer torchmetrics wants task= in accuracy(); this form works on both old and new.
patch("ctc/ctc_model.py", [("acc = accuracy(preds, y)", 'acc = accuracy(preds, y, task="multiclass", num_classes=logits.shape[1])')])

# cvit_cub.py: after fit(), Lightning 1.9 forgets the datamodule, so hand it over explicitly and test the best checkpoint.
patch("cvit_cub.py", [("test_results = trainer.test()", """import data as _data
test_results = trainer.test(model, datamodule=getattr(_data, args.data_name)(**vars(args)),
                            ckpt_path=None if args.debug else "best")""")])

# NumPy 2 dropped np.Inf and friends, but Lightning 1.9.5 still uses them, so put the aliases back.
if "import numpy as _np" not in pathlib.Path("ctc/__init__.py").read_text():
  patch("ctc/__init__.py", [("from .ctc import concepts_cost,", """import numpy as _np
for _old, _new in (("Inf", "inf"), ("NaN", "nan"), ("float_", "float64")):
    if not hasattr(_np, _old):
        setattr(_np, _old, getattr(_np, _new))

from .ctc import concepts_cost,""")])

# datamodule: newer albumentations moved these transforms, so import them from the top level.
patch("data/cub2011parts_datamodule.py", [
    ("from albumentations.augmentations.geometric.resize import Resize\n", "from albumentations import Resize\n"),
    ("from albumentations.augmentations.geometric.rotate import Rotate\n", "from albumentations import Rotate\n"),
    ("from albumentations.augmentations.transforms import HorizontalFlip, Normalize\n", "from albumentations import HorizontalFlip, Normalize\n"),
])

# Trainer branch for Apple GPUs (not used on Colab, harmless to keep).
patch("ctc/ctc_model.py", [("""        else:
            trainer = pl.Trainer(
                logger=logger,
                max_epochs=max_epochs,
                callbacks=callbacks,
            )""", """        else:
            extra = dict(accelerator="mps", devices=1) if torch.backends.mps.is_available() else {}
            trainer = pl.Trainer(
                logger=logger,
                max_epochs=max_epochs,
                callbacks=callbacks,
                gradient_clip_val=1.0,
                **extra,
            )""")])

# viz_utils.py: Pillow 10 fix, and let us point it at our own data folder.
patch("viz_utils.py", [("Image.ANTIALIAS", "Image.LANCZOS")])
# The explanation plot uses fixed thresholds (0.6 / 0.3 / 0.2) that suit a fully trained model. With flatter attention it
# can select nothing and crash, so use "the threshold, or at least the top few" instead.
patch("viz_utils.py", [
    ("patch_idx = attn.max(axis=1)[0] > 0.6",
     "mx = attn.max(axis=1)[0]\n    patch_idx = mx >= min(0.6, mx.topk(max(1, len(mx) // 10))[0][-1].item())"),
    ("attr_idx = attn[patch_idx,:].max(axis=0)[0] > 0.3",
     "av = attn[patch_idx,:].max(axis=0)[0]\n    attr_idx = av >= min(0.3, av.topk(min(5, len(av)))[0][-1].item())"),
    ("expl_idx = expl > 0.2",
     "expl_idx = expl >= min(0.2, expl.topk(min(3, len(expl)))[0][-1].item())"),
])
patch("viz_utils.py", [("def load_attributes(root='/data/Datasets/'):", "def load_attributes(root=os.environ.get('CUB_ROOT', '/data/Datasets/')):")])
!grep -n "progress_bar_refresh_rate\|weights_summary\|apex" ctc/ctc_model.py || echo "no leftover removed args"

### Download CUB-200-2011 and fix the one broken data file

In [ ]:
import os, tarfile, shutil
os.makedirs(DATA_DIR, exist_ok=True)
TGZ = f"{DATA_DIR}/CUB_200_2011.tgz"
URLS = ["https://data.caltech.edu/records/65de6-vp158/files/CUB_200_2011.tgz?download=1",
        "https://data.caltech.edu/records/65de6-vp158/files/CUB_200_2011.tgz"]

if not os.path.isdir(f"{DATA_DIR}/CUB_200_2011/images"):
    if not (os.path.exists(TGZ) and os.path.getsize(TGZ) > 1e9):
        for u in URLS:
            rc = os.system(f'curl -L --fail -o "{TGZ}" "{u}"')
            if rc == 0 and os.path.getsize(TGZ) > 1e9: break
        else:
            raise SystemExit("Download failed. Download CUB_200_2011.tgz manually from "
                             "https://www.vision.caltech.edu/datasets/cub_200_2011/ and upload it to " + DATA_DIR)
    with tarfile.open(TGZ) as t: t.extractall(DATA_DIR)

# attributes.txt comes out at the archive root, but the repo looks for it inside CUB_200_2011/
if os.path.exists(f"{DATA_DIR}/attributes.txt"):
    shutil.move(f"{DATA_DIR}/attributes.txt", f"{DATA_DIR}/CUB_200_2011/attributes.txt")
os.environ["CUB_ROOT"] = DATA_DIR

In [ ]:
attr_file = f"{DATA_DIR}/CUB_200_2011/attributes/image_attribute_labels.txt"
if not os.path.exists(attr_file + ".orig"):
    shutil.copy(attr_file, attr_file + ".orig")
bad = 0
with open(attr_file + ".orig") as fin, open(attr_file, "w") as fout:
    for i, line in enumerate(fin, 1):
        t = line.split()
        if len(t) != 5:
            bad += 1
            if bad <= 5: print(f"line {i}: {len(t)} fields -> {line.rstrip()!r}")
        if len(t) >= 4:
            t = (t + ["0"])[:5] if len(t) < 5 else t[:5]
            fout.write(" ".join(t) + "\n")
print("malformed lines fixed:", bad)

## Part 2: The script, `cvit_cub.py`, step by step

### 2.1 Imports and two constants

The script imports the model wrapper (`CTCModel`) and the function that runs a whole experiment (`run_exp`). `CTC_MODEL` picks the
CUB version of the Concept Transformer (a ViT backbone with the concept head), and `DATA_NAME` picks the CUB dataset loader with
body-part and attribute annotations.

In [ ]:
%cd {REPO_DIR}

from argparse import ArgumentParser
import numpy as np
import torch
from ctc import CTCModel, run_exp

CTC_MODEL = 'cub_cvit'
DATA_NAME = 'CUB2011Parts'

### 2.2 The command-line options

`get_parser` adds the CUB-specific options on top of the model's own. These are the knobs: learning rate, batch size, number of
epochs, warm-up, how strongly to weight the explanation loss (`--expl_lambda`), how strongly to push the attention to be sparse
(`--attention_sparsity`), and `--baseline` to train without concepts. This is the repo's code, unchanged.

In [ ]:
def get_parser(parser):
    parser = ArgumentParser(description='CUB dataset with explanations',
                            parents=[parser], conflict_handler='resolve')
    parser.add_argument('--learning_rate', default=0.00005, type=float)
    parser.add_argument('--batch_size', default=32, type=int)
    parser.add_argument('--debug', action='store_true',
                        help='Set debug mode in Lightning module')
    parser.add_argument('--data_dir', default='~/data/cub2011/', type=str,
                        help='dataset root directory')
    parser.add_argument('--weight_decay', default=1e-3, type=float,
                        help='weight decay (default: 1e-3)')
    parser.add_argument('--attention_sparsity', default=0.5, type=float,
                        help='sparsity penalty on attention')
    parser.add_argument('--max_epochs', default=50, type=int)
    parser.add_argument('--warmup', default=10, type=int, metavar='N',
                        help='number of warmup epochs')
    parser.add_argument('--finetune_unfreeze_epoch', default=0, type=int, metavar='N',
                        help='Epoch until when to finetune classifier head before unfreeezing feature extractor')
    parser.add_argument('--disable_lr_scheduler', action='store_true',
                        help='disable cosine lr schedule')
    parser.add_argument('--baseline', action='store_true',
                        help='run baseline without concepts')
    parser.add_argument('--expl_lambda', default=1.0, type=float,
                        help='weight of explanation loss')
    parser.add_argument('--num_workers', default=8, type=int, help='number of workers')
    parser.add_argument('--seed', default=42, type=int, help='random seed')
    return parser

### 2.3 Building the arguments

On the command line the script reads `sys.argv`. In a notebook there is no command line, so I hand the same options over as a list.
Here I pass the values from my settings cell. Add `"--baseline"` to the list if you want to train without concepts.

In [ ]:
parser = CTCModel.get_model_args()
parser = get_parser(parser)

args = parser.parse_args([
    "--data_dir", DATA_DIR,
    "--max_epochs", str(MAX_EPOCHS),
    "--batch_size", str(BATCH_SIZE),
    "--warmup", str(WARMUP),
    "--num_workers", str(NUM_WORKERS),
    "--learning_rate", str(LEARNING_RATE),
] + (["--debug"] if DEBUG else []))

args.ctc_model = CTC_MODEL
args.data_name = DATA_NAME
print(args)

### 2.4 Random seeds

So that a rerun gives the same split and starting weights.

In [ ]:
torch.manual_seed(args.seed)
np.random.seed(args.seed)

### 2.5 Train

`run_exp` does the heavy lifting: it builds the model and the data module, sets up a checkpoint that keeps the epoch with the lowest
validation loss, and trains with Lightning (fp16 on the GPU). On my A100 this took about 37 minutes for 50 epochs.

In [ ]:
model, trainer = run_exp(args)

### 2.6 Test

The original script ends with `trainer.test()`. In the old Lightning version that loaded the data on its own; in the version that
runs on Colab, Lightning forgets the data module after training, so I pass it in explicitly and test the best checkpoint (the one with
the lowest validation loss) instead of the last epoch.

In [ ]:
import data as _data

datamodule = getattr(_data, args.data_name)(**vars(args))
test_results = trainer.test(model, datamodule=datamodule, ckpt_path=None if args.debug else "best")
test_results

### What to look at next

- `test_acc` is the accuracy on the whole test set.
- `test_expl_loss` says how far the model's concept attention is from the annotated concepts, and `test_l1_loss` is the sparsity term.
- The checkpoint is saved under `cub_cvit/CUB2011Parts_expl1.0/`. My main notebook loads it to draw the concept explanations.